# Análises complementares – ECOS e paisagem virtual

Notebook separado do `paisagem_virtual_etapas.ipynb` (não depende dele). Rode as células em ordem.

- **D1.** Quantos pontos para estimar o estoque médio e para detectar diferença entre sistemas (dados observados)
- **D2.** Paisagem virtual condicionada aos pontos reais (o mapa passa pelos valores medidos)
- **D2b.** Figura da paisagem real × virtual em alta resolução (para o artigo)
- **D3.** Segunda rodada: sinal linear × não linear, várias realizações da paisagem
- **D4.** Sensibilidade ao alcance do variograma
- **D5.** Desenhos amostrais: aleatório, grade e guiado pela imagem
- **D6.** Pergunta invertida: com 20 a 26 pontos, qual o sinal mínimo para o modelo funcionar?

Siglas nas figuras (inglês): NV = native vegetation (VN), ICL = integrated crop-livestock (ILP), ILF = integrated livestock-forest (IPF).

**Tempo:** D3 a D6 são as etapas demoradas. Para um teste rápido, use `REPS = 3` e `REAL = 1`; para o resultado final, `REPS = 10` e `REAL = 2` (ou mais, se houver tempo).

In [ ]:
# %pip install rasterio geopandas gstools scikit-learn matplotlib scipy

## Configuração

In [ ]:
import os, warnings, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.enums import Resampling
from shapely import contains_xy
from scipy import stats
from scipy.stats import norm
from scipy.spatial import cKDTree
import gstools as gs
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
warnings.filterwarnings('ignore')

DADOS = os.path.join('..', 'data')            # pasta de dados do repositório
OBS = os.path.join(DADOS, 'soc_points.csv')
SAIDA = os.path.join('..', 'results', 'notebook_run')
os.makedirs(SAIDA, exist_ok=True)

# raster multiespectral de cada área (~1,25 m; bandas: azul, verde, pan, vermelho, red edge, NIR) e pontos amostrais
AREAS = {a: (os.path.join(DADOS, 'rasters', f'{a}_1m.tif'), os.path.join(DADOS, 'points', f'{a}_points.gpkg'))
         for a in ('VN', 'ILP', 'IPF')}
SIGLA_EN = {'VN': 'NV', 'ILP': 'ICL', 'IPF': 'ILF'}
NOME_EN = {'VN': 'Native vegetation (NV)', 'ILP': 'Integrated crop-livestock (ICL)', 'IPF': 'Integrated livestock-forest (ILF)'}
CAMADAS = ['solo0-10', 'solo10-20', 'solo20-30', 'solo0-30']
ROT = {'solo0-10': '0–10 cm', 'solo10-20': '10–20 cm', 'solo20-30': '20–30 cm', 'solo0-30': '0–30 cm'}
ALVO = 'solo0-30'            # camada usada nas simulações (D2 a D6)
PIX, MARGEM = 2.5, 15
NS = [10, 20, 30, 50, 75, 100, 150, 200, 300]
SINAIS = [0.3, 0.5, 0.7]     # força do sinal (R²) nos cenários
REPS = 10                    # amostras virtuais por combinação
REAL = 2                     # realizações (paisagens diferentes) por cenário
N_AVAL = 5000                # pixels usados para avaliar contra o mapa verdadeiro
FRAC, PROB = 0.8, 0.8        # sucesso: R² >= 80% do teto em pelo menos 80% das repetições
SEED = 2024
IDX = ['EVI', 'SAVI', 'NDVI', 'NDRE', 'PRI', 'sPRI', 'CO2FLUX', 'NDWI', 'SOCI', 'SI', 'CI', 'OI']
rng = np.random.default_rng(SEED)
obs_all = pd.read_csv(OBS)

mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.linewidth': .6,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 400,
                     'axes.titlesize': 7.5, 'legend.fontsize': 6.5})
COR = {'VN': '#2a78d6', 'ILP': '#eb6834', 'IPF': '#1baf7a'}
INK, MUTED = '#222222', '#6b6b6b'

## Funções

In [ ]:
def indices(B, G, R, RE, N):
    ndvi = (N - R) / (N + R); pri = (G - B) / (G + B); spri = (pri + 1) / 2
    return {'EVI': 2.5 * (N - R) / (N + 6 * R - 7.5 * B + 1), 'SAVI': 1.5 * (N - R) / (N + R + 0.5),
            'NDVI': ndvi, 'NDRE': (N - RE) / (N + RE), 'PRI': pri, 'sPRI': spri,
            'CO2FLUX': ndvi * spri, 'NDWI': (G - N) / (G + N), 'SOCI': B / (R * G),
            'SI': (R - B) / (R + B), 'CI': (R - G) / (R + G), 'OI': (2 * R - G - B) / (R + G)}

def anamorfose(z, alvo):
    # valores normais padrão -> distribuição empírica dos observados
    q = np.sort(alvo); p = (np.arange(1, len(q) + 1) - 0.5) / len(q)
    return np.interp(norm.cdf(z), p, q)

def escore_normal(y):
    # observados -> valores normais padrão (inverso da anamorfose)
    return norm.ppf((stats.rankdata(y) - 0.5) / len(y))

def padroniza(v):
    return (v - v.mean()) / v.std()

def teto_r2(s, y, nbins=100):
    # R² máximo possível: parte da variância de y explicada pelo sinal s (razão de correlação, η²).
    # Vale para sinal linear ou não linear (um modelo perfeito prediria a média de y para cada valor de s).
    b = np.minimum((stats.rankdata(s) - 1) * nbins // len(s), nbins - 1).astype(int)
    m = np.bincount(b, y) / np.bincount(b)
    return 1 - np.var(y - m[b]) / np.var(y)

def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 30)))

def floresta(seed=0):
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=seed, n_jobs=1)

def n_minimo(d, col='sucesso'):
    # menor n com P(sucesso) >= PROB; '>300' se nenhum
    p = d.groupby('n')[col].mean()
    ok = p[p >= PROB]
    return str(int(ok.index.min())) if len(ok) else f'>{max(NS)}'

## D1. Quantos pontos para estimar a média e detectar diferenças (dados observados)
- **Média:** n para estimar o estoque médio de cada área com erro relativo de 5, 10 e 20% (95% de confiança), `n = (t·CV/E)²`, resolvido iterativamente. Supõe pontos independentes; com dependência espacial o n real é um pouco maior.
- **Diferença entre sistemas:** n por área para detectar diferenças de 5 a 30 Mg ha⁻¹ (teste t, α = 0,05, poder = 0,80), com o desvio padrão combinado de cada par.
- **Diferença mínima detectável** com o n atual (20 a 26 pontos).

In [ ]:
def n_media(cv, erro, conf=0.95):
    n = 2
    for _ in range(100):
        t = stats.t.ppf(1 - (1 - conf) / 2, max(n - 1, 1)); novo = int(np.ceil((t * cv / erro) ** 2))
        if novo == n: break
        n = max(novo, 2)
    return n

def poder_t(n, delta, sd, alfa=0.05):
    df = 2 * n - 2; nc = delta / (sd * np.sqrt(2 / n)); tc = stats.t.ppf(1 - alfa / 2, df)
    return stats.nct.sf(tc, df, nc) + stats.nct.cdf(-tc, df, nc)

def n_diferenca(delta, sd, alvo=0.8):
    for n in range(2, 5001):
        if poder_t(n, delta, sd) >= alvo: return n
    return np.nan

def mdd(n, sd, alvo=0.8):
    return next(d for d in np.arange(0.1, 200, 0.1) if poder_t(n, d, sd) >= alvo)

med = []
for (a, c), g in obs_all.melt(id_vars='tratamento', value_vars=CAMADAS, var_name='camada').groupby(['tratamento', 'camada']):
    y = g.value.values; cv = y.std(ddof=1) / y.mean() * 100
    med.append(dict(area=a, camada=ROT[c], n_atual=len(y), media=y.mean(), CV=cv,
                    erro_atual_pct=stats.t.ppf(.975, len(y) - 1) * cv / np.sqrt(len(y)),
                    **{f'n_erro_{e}%': n_media(cv, e) for e in (5, 10, 20)}))
D1a = pd.DataFrame(med).round(1)
D1a['area'] = pd.Categorical(D1a.area, ['VN', 'ILP', 'IPF']); D1a = D1a.sort_values(['area', 'camada'])
D1a.to_csv(os.path.join(SAIDA, 'D1a_n_para_media.csv'), index=False)

dif = []
for c in CAMADAS:
    for a, b in [('VN', 'ILP'), ('VN', 'IPF'), ('ILP', 'IPF')]:
        ya, yb = (obs_all.loc[obs_all.tratamento == k, c].values for k in (a, b))
        sd = np.sqrt((ya.var(ddof=1) * (len(ya) - 1) + yb.var(ddof=1) * (len(yb) - 1)) / (len(ya) + len(yb) - 2))
        n0 = int(round((len(ya) + len(yb)) / 2))
        dif.append(dict(camada=ROT[c], par=f'{a}×{b}', dif_observada=abs(ya.mean() - yb.mean()), sd_comb=sd,
                        MDD_n_atual=mdd(n0, sd), **{f'n_dif_{d}': n_diferenca(d, sd) for d in (5, 10, 20, 30)}))
D1b = pd.DataFrame(dif).round(1); D1b.to_csv(os.path.join(SAIDA, 'D1b_n_para_diferencas.csv'), index=False)
display(D1a); display(D1b)

Figura D1 (inglês): erro relativo esperado da média em função do n, por área e camada.

In [ ]:
ns = np.arange(5, 201)
fig, ax = plt.subplots(1, 4, figsize=(7.2, 2.3), sharey=True, constrained_layout=True)
for A, c in zip(ax, CAMADAS):
    for a in ['VN', 'ILP', 'IPF']:
        y = obs_all.loc[obs_all.tratamento == a, c]; cv = y.std() / y.mean() * 100
        A.plot(ns, stats.t.ppf(.975, ns - 1) * cv / np.sqrt(ns), color=COR[a], lw=1.3, label=NOME_EN[a])
        A.plot(len(y), stats.t.ppf(.975, len(y) - 1) * cv / np.sqrt(len(y)), 'o', ms=3.5, color=COR[a], mec='white', mew=.6)
    A.axhline(10, color=MUTED, lw=.6, ls='--'); A.set_xscale('log'); A.set_xticks([5, 10, 20, 50, 100, 200]); A.set_xticklabels([5, 10, 20, 50, 100, 200])
    A.minorticks_off(); A.set_ylim(0, 40); A.set_title(ROT[c], loc='left'); A.set_xlabel('Number of samples'); A.grid(color='#e2e2df', lw=.5)
ax[0].set_ylabel('Relative error of the mean (%, 95% CI)')
ax[-1].legend(frameon=False, fontsize=5.5, loc='upper right')
fig.suptitle('Sample size needed to estimate the mean SOC stock (dots = this study; dashed line = ±10%)', fontsize=7.5)
for e in ('png', 'pdf'): fig.savefig(os.path.join(SAIDA, f'D1_sample_size_mean.{e}'))
plt.show()

## D2. Paisagem virtual condicionada aos pontos reais
Lê os ortomosaicos (2,5 m), calcula os índices e as duas primeiras componentes principais (PC1, PC2), que resumem o gradiente espectral dominante. Os sinais simulados são:
- **linear:** o ECOS acompanha a PC1;
- **não linear:** o ECOS responde a PC1 com saturação, a PC2 de forma quadrática e à interação PC1 × PC2 (favorece modelos como o RF).

O resíduo espacial é um campo gaussiano com o variograma ajustado nos pontos. Há duas versões:
- **condicionada** (`condicionar=True`): o mapa passa pelos estoques medidos. Usada no cenário observado (sem sinal) e na figura D2b.
- **não condicionada** (padrão): usada nos cenários com sinal (D3 a D6). Os pontos reais não têm sinal espectral, então condicionar a eles um mapa com sinal forte é contraditório: o condicionamento "apaga" o sinal (na VN, o R² máximo pedido de 0,5 cai para ~0,35). Os cenários com sinal respondem "e se a paisagem tivesse esse sinal?", mantendo a distribuição e o variograma observados.

In [ ]:
PAIS = {}
for area, (raster, pontos) in AREAS.items():
    g = gpd.read_file(pontos).to_crs(31983)
    dom = g.union_all().convex_hull.buffer(MARGEM)
    with rasterio.open(raster) as src:
        win = rasterio.windows.from_bounds(*dom.bounds, transform=src.transform)
        f = src.res[0] / PIX; h, w = int(win.height * f), int(win.width * f)
        a = src.read(window=win, out_shape=(6, h, w), resampling=Resampling.average, masked=True, boundless=True)
        tr = src.window_transform(win) * rasterio.Affine.scale(win.width / w, win.height / h)
    cols, rows = np.meshgrid(np.arange(w), np.arange(h))
    xs, ys = (np.asarray(v) for v in rasterio.transform.xy(tr, rows.ravel(), cols.ravel()))
    B, G, _, R, RE, N = [a[i].filled(np.nan).ravel().astype(float) for i in range(6)]
    ok = contains_xy(dom, xs, ys) & np.isfinite(B) & (B > 0) & (N > 0)
    I = pd.DataFrame(indices(B[ok], G[ok], R[ok], RE[ok], N[ok])); I['x'], I['y'] = xs[ok], ys[ok]
    I = I.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    X = StandardScaler().fit_transform(I[IDX].values)
    pc = PCA(2, random_state=0).fit_transform(X)
    pc = np.column_stack([escore_normal(pc[:, j]) for j in range(2)])   # escores normais: evita que pixels extremos dominem o sinal
    sinais = {'linear': padroniza(pc[:, 0]),
              'nonlinear': padroniza(np.tanh(1.5 * pc[:, 0]) + 0.6 * pc[:, 1] ** 2 - 0.5 * pc[:, 0] * pc[:, 1])}
    _, k = cKDTree(I[['x', 'y']].values).query(np.c_[g.geometry.x, g.geometry.y])
    obs = obs_all[obs_all.tratamento == area].reset_index(drop=True)
    yo = obs[ALVO].values; zo = escore_normal(yo); pos = np.c_[g.geometry.x, g.geometry.y]
    try:   # variograma dos escores normais nos pontos
        bins = np.linspace(0, np.ptp(pos, 0).max() * 0.6, 8)
        bc, gam = gs.vario_estimate((pos[:, 0], pos[:, 1]), zo, bins)
        mod = gs.Exponential(dim=2); mod.fit_variogram(bc, gam, nugget=True)
        alc, pep = float(np.clip(mod.len_scale, 10, 300)), float(np.clip(mod.nugget / (mod.nugget + mod.var), 0, .9))
    except Exception:
        alc, pep = 50.0, 0.3
    PAIS[area] = dict(I=I, X=I[IDX].values, sinais=sinais, k=k, pos=pos, yo=yo, zo=zo, alc=alc, pep=pep)
    print(f'{area}: {len(I)} pixels ({len(I) * PIX**2 / 1e4:.1f} ha); PCA explica '
          f'{PCA(2).fit(X).explained_variance_ratio_.sum():.0%}; alcance {alc:.0f} m, pepita relativa {pep:.2f}')

def paisagem(P, tipo, r2, seed, alcance=None, condicionar=False, calibrar=True):
    """ECOS virtual em cada pixel. Retorna (y, teto).
    O condicionamento aos pontos (onde o sinal observado é ~0) reduz o sinal efetivo; com calibrar=True
    o R² de entrada é ajustado até o teto realizado ficar próximo do R² pedido."""
    r_in = r2
    for _ in range(8 if calibrar else 1):
        y, teto = _paisagem(P, tipo, r_in, seed, alcance, condicionar)
        if not calibrar or abs(teto - r2) < 0.03: break
        r_in = float(np.clip(r_in * r2 / max(teto, 1e-3), 0.01, 0.97))
    return y, teto

def _paisagem(P, tipo, r2, seed, alcance, condicionar):
    I, s = P['I'], P['sinais'][tipo]
    alc = P['alc'] if alcance is None else alcance
    cov = gs.Exponential(dim=2, var=1 - P['pep'], len_scale=alc, nugget=P['pep'])
    if condicionar:   # resíduo nos pontos para que z = sqrt(r2)*s + sqrt(1-r2)*e reproduza os escores observados
        e_pts = (P['zo'] - np.sqrt(r2) * s[P['k']]) / np.sqrt(1 - r2)
        krig = gs.krige.Ordinary(cov, cond_pos=(P['pos'][:, 0], P['pos'][:, 1]), cond_val=e_pts, exact=False)
        e = gs.CondSRF(krig, seed=seed)((I.x.values, I.y.values), mesh_type='unstructured')
    else:
        e = gs.SRF(cov, seed=seed)((I.x.values, I.y.values), mesh_type='unstructured')
    z = np.sqrt(r2) * s + np.sqrt(1 - r2) * e
    y = anamorfose(padroniza(z), P['yo'])
    return y, teto_r2(s, y)

# conferência: o mapa condicionado (cenário observado) passa perto dos valores medidos?
for area, P in PAIS.items():
    y, teto = paisagem(P, 'linear', 0.0, seed=1, condicionar=True, calibrar=False)
    print(f"{area}: r entre medido e mapa nos pontos = {np.corrcoef(P['yo'], y[P['k']])[0, 1]:.2f}")

## D2b. Figura: paisagem real × virtual (versão para o artigo)
- Painel real: ortomosaico lido com pixel de 0,5 m (`PIX_RGB`), cores das bandas vermelho, verde e azul do RedEdge-P, contraste esticado (percentis 2–98, o mesmo nas três bandas, para preservar as cores) e ajuste gama.
- Painel virtual: uma realização condicionada aos pontos no cenário observado (sem sinal espectral), pixel de 2,5 m.
- Pontos coloridos pelo estoque medido nos dois painéis, mesma escala de cor.

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator
import scipy.ndimage as ndi
PIX_RGB, GAMA = 0.5, 0.85
SOC = LinearSegmentedColormap.from_list('soc', ['#f6efe3', '#d9b98a', '#a8743f', '#6b4423', '#3b2412'])
fmt = FuncFormatter(lambda v, _: f'{v:,.0f}'.replace(',', ' '))
vmin, vmax = np.percentile(obs_all[ALVO], [2, 98])

def grade_pix(I, vals):
    ix = np.round((I.x - I.x.min()) / PIX).astype(int); iy = np.round((I.y.max() - I.y) / PIX).astype(int)
    out = np.full((iy.max() + 1, ix.max() + 1), np.nan); out[iy, ix] = vals
    ok = np.zeros(out.shape, bool); ok[iy, ix] = True
    buraco = ndi.binary_closing(ok, np.ones((3, 3))) & ~ok   # falhas de 1 pixel no mosaico
    _, (ri, ci) = ndi.distance_transform_edt(~ok, return_indices=True)
    out[buraco] = out[ri[buraco], ci[buraco]]; return out

FIG = {}
for area, (raster, pontos) in AREAS.items():
    P = PAIS[area]; g = gpd.read_file(pontos).to_crs(31983)
    dom = g.union_all().convex_hull.buffer(MARGEM)
    with rasterio.open(raster) as src:
        win = rasterio.windows.from_bounds(*dom.bounds, transform=src.transform)
        f = src.res[0] / PIX_RGB; h, w = int(win.height * f), int(win.width * f)
        b = src.read([4, 2, 1], window=win, out_shape=(3, h, w), resampling=Resampling.average,
                     masked=True, boundless=True).filled(np.nan).astype(float)
        tr = src.window_transform(win) * rasterio.Affine.scale(win.width / w, win.height / h)
    cols, rows = np.meshgrid(np.arange(w), np.arange(h))
    xs, ys = (np.asarray(v) for v in rasterio.transform.xy(tr, rows.ravel(), cols.ravel()))
    dentro = contains_xy(dom, xs, ys).reshape(h, w) & np.isfinite(b).all(0) & (b > 0).all(0)
    rgb = np.zeros((h, w, 4))
    lo, hi = np.nanpercentile(b[:, dentro], [2, 98])   # mesmo estiramento nas 3 bandas (preserva as cores)
    for i in range(3):
        rgb[..., i] = np.clip((b[i] - lo) / (hi - lo), 0, 1) ** GAMA
    rgb[..., 3] = dentro
    y0, _ = paisagem(P, 'linear', 0.0, seed=7, condicionar=True, calibrar=False)   # cenário observado, condicionado aos pontos
    I = P['I']
    FIG[area] = dict(g=g, rgb=rgb, ext_rgb=[tr.c, tr.c + w * tr.a, tr.f + h * tr.e, tr.f], virt=grade_pix(I, y0),
                     ext_v=[I.x.min() - PIX / 2, I.x.max() + PIX / 2, I.y.min() - PIX / 2, I.y.max() + PIX / 2])
    e = FIG[area]['ext_rgb']; FIG[area]['asp'] = (e[3] - e[2]) / (e[1] - e[0])

hr = [max(d['asp'], .35) for d in FIG.values()]
fig, ax = plt.subplots(3, 2, figsize=(7.2, 7.2 * .5 * sum(hr) + 1.0), gridspec_kw=dict(height_ratios=hr), constrained_layout=True)
for i, (area, d) in enumerate(FIG.items()):
    x, yy, yo = d['g'].geometry.x, d['g'].geometry.y, PAIS[area]['yo']
    ax[i, 0].imshow(d['rgb'], extent=d['ext_rgb'], interpolation='antialiased')
    im = ax[i, 1].imshow(d['virt'], extent=d['ext_v'], cmap=SOC, vmin=vmin, vmax=vmax, interpolation='nearest')
    for j, A in enumerate(ax[i]):
        A.scatter(x, yy, c=yo, cmap=SOC, vmin=vmin, vmax=vmax, s=24 if j == 0 else 14,
                  edgecolors='#ffffff' if j == 0 else INK, linewidths=.8 if j == 0 else .5, zorder=3)
        A.set_xlim(d['ext_rgb'][:2]); A.set_ylim(d['ext_rgb'][2:]); A.set_aspect('equal')
        A.xaxis.set_major_formatter(fmt); A.yaxis.set_major_formatter(fmt)
        A.xaxis.set_major_locator(MaxNLocator(4)); A.yaxis.set_major_locator(MaxNLocator(3)); A.tick_params(labelsize=5.5, length=2)
        if j == 1: A.set_yticklabels([])
        tit = f'{NOME_EN[area]}: real landscape' if j == 0 else 'Virtual landscape (simulated SOC stock)'
        A.set_title(f"{'abcdef'[i * 2 + j]}   {tit}", loc='left')
    ax[i, 0].set_ylabel('Northing (m)', fontsize=6)
for A in ax[2]: A.set_xlabel('Easting (m)', fontsize=6)
cb = fig.colorbar(im, ax=ax, orientation='horizontal', shrink=.5, aspect=35, pad=.01)
cb.set_label(f'SOC stock {ROT[ALVO]} (Mg ha⁻¹)'); cb.outline.set_visible(False)
fig.suptitle('Real landscape (UAV multispectral mosaic, red-green-blue bands, 0.5 m; dots = measured SOC) and virtual landscape\n'
             '(SOC simulated at 2.5 m, conditioned on the field points, with the observed distribution and variogram). SIRGAS 2000 / UTM 23S', fontsize=6.5)
for e in ('png', 'pdf'): fig.savefig(os.path.join(SAIDA, f'D2b_real_vs_virtual_landscape.{e}'))
plt.show()

## D3. Segunda rodada: sinal linear × não linear
Para cada área, tipo de sinal, força do sinal e realização da paisagem: sorteia `n` pixels (aleatório), treina Ridge e RF e avalia contra o mapa verdadeiro (5000 pixels).
A força do sinal é definida na escala normal (antes da anamorfose); o R² máximo realizado na escala do estoque (`teto`) é calculado em cada paisagem e usado no critério de sucesso.

In [ ]:
def avalia(P, y, idx, aval, seed=0, modelos=('Ridge', 'RF')):
    out = {}
    for m in modelos:
        mdl = ridge() if m == 'Ridge' else floresta(seed)
        mdl.fit(P['X'][idx], y[idx]); out[m] = r2_score(y[aval], mdl.predict(P['X'][aval]))
    return out

t0 = time.time(); lin = []
for area, P in PAIS.items():
    for tipo in ['linear', 'nonlinear']:
        for r2 in SINAIS:
            for real in range(REAL):
                y, teto = paisagem(P, tipo, r2, seed=int(rng.integers(1e9)))
                aval = rng.choice(len(y), min(N_AVAL, len(y)), replace=False)
                for n in NS:
                    for rep in range(REPS):
                        idx = rng.choice(len(y), n, replace=False)
                        for m, v in avalia(P, y, idx, aval, rep).items():
                            lin.append(dict(area=area, sinal=tipo, R2_alvo=r2, realizacao=real, n=n, rep=rep,
                                            modelo=m, R2=v, teto=teto, sucesso=v >= FRAC * teto))
    print(area, f'{(time.time() - t0) / 60:.1f} min')
D3 = pd.DataFrame(lin); D3.to_csv(os.path.join(SAIDA, 'D3_simulacao_segunda_rodada.csv'), index=False)
D3n = (D3.groupby(['area', 'sinal', 'R2_alvo', 'modelo']).apply(n_minimo).rename('n_min').reset_index()
       .pivot_table(index=['sinal', 'R2_alvo'], columns=['area', 'modelo'], values='n_min', aggfunc='first')
       .reindex(columns=['VN', 'ILP', 'IPF'], level=0))
D3n.to_csv(os.path.join(SAIDA, 'D3_n_minimo.csv')); D3n

Figura D3 (inglês): curvas de aprendizado por área, sinal e modelo.

In [ ]:
cores = {0.3: '#9ecae1', 0.5: '#3182bd', 0.7: '#08519c'}
fig, ax = plt.subplots(2, 6, figsize=(7.2, 3.4), sharex=True, sharey=True, constrained_layout=True)
for j, (area, tipo) in enumerate([(a, t) for a in ['VN', 'ILP', 'IPF'] for t in ['linear', 'nonlinear']]):
    for i, m in enumerate(['Ridge', 'RF']):
        A = ax[i, j]
        for r2 in SINAIS:
            d = D3[(D3.area == area) & (D3.sinal == tipo) & (D3.modelo == m) & (D3.R2_alvo == r2)].groupby('n').R2
            A.plot(NS, d.median().reindex(NS), '-o', ms=2, lw=1.1, color=cores[r2], label=f'R² = {r2}')
            A.fill_between(NS, d.quantile(.1).reindex(NS), d.median().reindex(NS), color=cores[r2], alpha=.12, lw=0)
        A.axhline(0, ls='--', lw=.5, color=INK); A.set_xscale('log'); A.set_xticks([10, 30, 100, 300]); A.set_xticklabels([10, 30, 100, 300])
        A.minorticks_off(); A.set_ylim(-0.4, 0.8); A.grid(color='#e2e2df', lw=.5)
        if i == 0: A.set_title(f"{SIGLA_EN[area]} · {'linear' if tipo == 'linear' else 'non-linear'}", loc='left', fontsize=6.5)
        if j == 0: A.set_ylabel(f"{'Ridge regression' if m == 'Ridge' else 'Random forest'}\nR² against the true map")
        if i == 1: A.set_xlabel('Sample size (n)')
ax[0, -1].legend(title='Signal strength', fontsize=5, title_fontsize=5.5, frameon=False)
fig.suptitle(f'Learning curves in the virtual landscapes, linear and non-linear signal (median and 10th percentile, {REAL} landscapes × {REPS} samples)', fontsize=7)
for e in ('png', 'pdf'): fig.savefig(os.path.join(SAIDA, f'D3_learning_curves_linear_nonlinear.{e}'))
plt.show()

## D4. Sensibilidade ao alcance do variograma
O alcance ajustado com 20 a 26 pontos é a parte mais incerta da simulação. Aqui ele é fixado em 25, 75 e 200 m (sinal linear, R² = 0,5, só Ridge) para ver se o n mínimo muda.

In [ ]:
lin = []
for area, P in PAIS.items():
    for alc in [25, 75, 200]:
        for real in range(REAL):
            y, teto = paisagem(P, 'linear', 0.5, seed=int(rng.integers(1e9)), alcance=alc)
            aval = rng.choice(len(y), min(N_AVAL, len(y)), replace=False)
            for n in NS:
                for rep in range(REPS):
                    v = avalia(P, y, rng.choice(len(y), n, replace=False), aval, rep, ('Ridge',))['Ridge']
                    lin.append(dict(area=area, alcance=alc, realizacao=real, n=n, rep=rep, R2=v, teto=teto, sucesso=v >= FRAC * teto))
D4 = pd.DataFrame(lin); D4.to_csv(os.path.join(SAIDA, 'D4_sensibilidade_alcance.csv'), index=False)
D4n = D4.groupby(['area', 'alcance']).apply(n_minimo).unstack('area').reindex(columns=['VN', 'ILP', 'IPF'])
D4n.to_csv(os.path.join(SAIDA, 'D4_n_minimo_por_alcance.csv')); D4n

## D5. Desenhos amostrais
Compara três formas de escolher os `n` pontos (sinal R² = 0,5, linear e não linear):
- **aleatório** simples;
- **grade** regular com origem sorteada (como no campo);
- **guiado pela imagem**: agrupa os pixels em `n` grupos no espaço dos índices (k-médias) e coleta o pixel mais próximo do centro de cada grupo (*feature space coverage sampling*, Brus, 2019).

Avalia o R² do modelo contra o mapa verdadeiro e o erro do estoque médio da área estimado pela média simples dos pontos.

In [ ]:
def grade(I, n, rng):
    xy = I[['x', 'y']].values; area = len(I) * PIX ** 2; d = np.sqrt(area / n)
    for _ in range(50):
        ox, oy = rng.uniform(0, d, 2)
        gx, gy = np.meshgrid(np.arange(xy[:, 0].min() + ox, xy[:, 0].max(), d), np.arange(xy[:, 1].min() + oy, xy[:, 1].max(), d))
        dist, k = cKDTree(xy).query(np.c_[gx.ravel(), gy.ravel()])
        k = np.unique(k[dist < PIX])
        if abs(len(k) - n) <= max(1, n // 10): return k
        d *= np.sqrt(len(k) / n) if len(k) else 0.9
    return k

def guiado(P, n, rng):
    Xs = StandardScaler().fit_transform(P['X']); sub = rng.choice(len(Xs), min(20000, len(Xs)), replace=False)
    km = KMeans(n, n_init=1, random_state=int(rng.integers(1e6))).fit(Xs[sub])
    _, k = cKDTree(Xs).query(km.cluster_centers_)
    return np.unique(k)

NS5 = [10, 20, 30, 50, 100]
lin = []
for area, P in PAIS.items():
    for tipo in ['linear', 'nonlinear']:
        for real in range(REAL):
            y, teto = paisagem(P, tipo, 0.5, seed=int(rng.integers(1e9)))
            aval = rng.choice(len(y), min(N_AVAL, len(y)), replace=False)
            for n in NS5:
                for rep in range(REPS):
                    for des, fn in (('random', lambda: rng.choice(len(y), n, replace=False)),
                                    ('grid', lambda: grade(P['I'], n, rng)), ('image-guided', lambda: guiado(P, n, rng))):
                        idx = fn()
                        for m, v in avalia(P, y, idx, aval, rep).items():
                            lin.append(dict(area=area, sinal=tipo, realizacao=real, n=n, rep=rep, desenho=des, modelo=m, R2=v,
                                            teto=teto, sucesso=v >= FRAC * teto, erro_media_pct=abs(y[idx].mean() / y.mean() - 1) * 100))
D5 = pd.DataFrame(lin); D5.to_csv(os.path.join(SAIDA, 'D5_desenhos_amostrais.csv'), index=False)
D5r = D5.groupby(['area', 'sinal', 'desenho', 'modelo', 'n']).agg(R2_mediana=('R2', 'median'), P_sucesso=('sucesso', 'mean'),
                                                                   erro_media_p90=('erro_media_pct', lambda v: np.percentile(v, 90))).round(2).reset_index()
D5r.to_csv(os.path.join(SAIDA, 'D5_resumo.csv'), index=False)
D5r[(D5r.n == 20)]

Figura D5 (inglês): R² do modelo (RF) e erro da média por desenho amostral.

In [ ]:
cd = {'random': '#7f7f7f', 'grid': '#eb6834', 'image-guided': '#2a78d6'}
fig, ax = plt.subplots(2, 6, figsize=(7.2, 3.4), sharex=True, constrained_layout=True)
for j, (area, tipo) in enumerate([(a, t) for a in ['VN', 'ILP', 'IPF'] for t in ['linear', 'nonlinear']]):
    d = D5r[(D5r.area == area) & (D5r.sinal == tipo) & (D5r.modelo == 'RF')]
    for des in cd:
        e = d[d.desenho == des]
        ax[0, j].plot(e.n, e.R2_mediana, '-o', ms=2, lw=1.1, color=cd[des], label=des)
        ax[1, j].plot(e.n, e.erro_media_p90, '-o', ms=2, lw=1.1, color=cd[des])
    ax[0, j].set_title(f"{SIGLA_EN[area]} · {'linear' if tipo == 'linear' else 'non-linear'}", loc='left', fontsize=6.5)
    for A in ax[:, j]:
        A.set_xscale('log'); A.set_xticks(NS5); A.set_xticklabels(NS5); A.minorticks_off(); A.grid(color='#e2e2df', lw=.5)
    ax[0, j].set_ylim(-0.2, 0.6); ax[1, j].set_ylim(0, 25); ax[1, j].set_xlabel('Sample size (n)')
ax[0, 0].set_ylabel('Median R² (random forest)'); ax[1, 0].set_ylabel('Error of the mean (%, 90th pct.)')
ax[0, -1].legend(fontsize=5.5, frameon=False)
fig.suptitle('Sampling designs in the virtual landscape (signal R² = 0.5)', fontsize=7.5)
for e in ('png', 'pdf'): fig.savefig(os.path.join(SAIDA, f'D5_sampling_designs.{e}'))
plt.show()

## D6. Pergunta invertida: com o n do campo, qual o sinal mínimo?
Fixa `n` no número de pontos coletados em cada área (20, 26, 20) e varia a força do sinal de 0,1 a 0,9. O **sinal mínimo** é o menor R² em que pelo menos 80% das amostras atingem 80% do teto.

In [ ]:
R2S = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
lin = []
for area, P in PAIS.items():
    n = len(P['yo'])
    for tipo in ['linear', 'nonlinear']:
        for r2 in R2S:
            for real in range(REAL):
                y, teto = paisagem(P, tipo, r2, seed=int(rng.integers(1e9)))
                aval = rng.choice(len(y), min(N_AVAL, len(y)), replace=False)
                for rep in range(REPS):
                    for m, v in avalia(P, y, rng.choice(len(y), n, replace=False), aval, rep).items():
                        lin.append(dict(area=area, n=n, sinal=tipo, R2_alvo=r2, realizacao=real, rep=rep, modelo=m, R2=v,
                                        teto=teto, sucesso=v >= FRAC * teto))
D6 = pd.DataFrame(lin); D6.to_csv(os.path.join(SAIDA, 'D6_sinal_minimo.csv'), index=False)
def sinal_min(d):
    p = d.groupby('R2_alvo').sucesso.mean(); ok = p[p >= PROB]
    return f'{ok.index.min():.1f}' if len(ok) else '>0.9'
D6s = D6.groupby(['area', 'sinal', 'modelo']).apply(sinal_min).unstack('area').reindex(columns=['VN', 'ILP', 'IPF'])
D6s.to_csv(os.path.join(SAIDA, 'D6_sinal_minimo_tabela.csv')); D6s

Figura D6 (inglês): probabilidade de sucesso em função da força do sinal, com o n do campo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6), sharey=True, constrained_layout=True)
for A, tipo in zip(ax, ['linear', 'nonlinear']):
    for area in ['VN', 'ILP', 'IPF']:
        for m, ls in (('Ridge', '-'), ('RF', '--')):
            p = D6[(D6.area == area) & (D6.sinal == tipo) & (D6.modelo == m)].groupby('R2_alvo').sucesso.mean()
            A.plot(p.index, p.values, ls, marker='o', ms=2.5, lw=1.2, color=COR[area],
                   label=f"{SIGLA_EN[area]} (n = {len(PAIS[area]['yo'])}) · {'Ridge' if m == 'Ridge' else 'RF'}")
    A.axhline(PROB, color=MUTED, lw=.6, ls=':'); A.set_ylim(0, 1.02); A.grid(color='#e2e2df', lw=.5)
    A.set_title('Linear signal' if tipo == 'linear' else 'Non-linear signal', loc='left'); A.set_xlabel('Strength of the spectral signal (R²)')
ax[0].set_ylabel('Probability of reaching 80% of attainable R²')
ax[1].legend(fontsize=5.3, frameon=False, loc='lower right', ncol=2)
fig.suptitle('How strong must the signal be for the field sample size of this study?', fontsize=7.5)
for e in ('png', 'pdf'): fig.savefig(os.path.join(SAIDA, f'D6_minimum_signal.{e}'))
plt.show()